# Week 1 示例：NumPy 向量化练习

- 作者：共享仓库示例
- 日期：2026-07-15
- 来源：《实验室新生暑期居家集训学习计划》Week 1，NumPy 核心
- 适用周次：Week 1
- 分类：NumPy / SciPy
- 关键词：广播、z-score、余弦相似度、softmax、one-hot、向量化 k-NN
- 运行环境：Python 3.10+、NumPy；SciPy 为可选依赖

## 学习目标

1. 用广播完成批量数组运算。
2. 不使用显式 `for` 循环实现几个常见的机器学习基础操作。
3. 为函数增加简单的形状检查和 `assert` 自检。

> 这是从原始学习计划五道 NumPy 练习中抽取的可运行示例，不替代个人完整练习。

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
print('NumPy:', np.__version__)

# 广播：每一行减去自己的均值，结果仍保持二维形状。
data = rng.normal(size=(4, 3))
row_means = data.mean(axis=1, keepdims=True)
centered = data - row_means
print('data shape:', data.shape)
print('centered row means:', centered.mean(axis=1).round(8))
assert centered.shape == data.shape
assert np.allclose(centered.mean(axis=1), 0)

## 1. 五个向量化函数

这些函数对应原始计划中的 z-score、余弦相似度、数值稳定版 softmax、one-hot 和 k-NN 距离矩阵练习。关键点是利用 NumPy 的广播和矩阵乘法表达批量计算。

In [ ]:
def z_score(x):
    """按列标准化，保留二维形状。"""
    x = np.asarray(x, dtype=np.float64)
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    return (x - mean) / np.maximum(std, 1e-12)


def cosine_similarity_matrix(x):
    """返回 x 中所有向量的两两余弦相似度。"""
    x = np.asarray(x, dtype=np.float64)
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    normalized = x / np.maximum(norms, 1e-12)
    return normalized @ normalized.T


def softmax(x):
    """沿最后一维计算数值稳定的 softmax。"""
    x = np.asarray(x, dtype=np.float64)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=-1, keepdims=True)


def one_hot(labels, num_classes):
    labels = np.asarray(labels, dtype=np.int64)
    result = np.zeros((labels.size, num_classes), dtype=np.float32)
    result[np.arange(labels.size), labels] = 1.0
    return result


def knn_distances(query, database):
    """用广播返回 query 与 database 的欧氏距离矩阵。"""
    query = np.asarray(query, dtype=np.float64)
    database = np.asarray(database, dtype=np.float64)
    squared = ((query[:, None, :] - database[None, :, :]) ** 2).sum(axis=-1)
    return np.sqrt(squared)


In [ ]:
features = rng.normal(size=(10, 4))
normalized = z_score(features)
similarities = cosine_similarity_matrix(features)
probabilities = softmax(np.array([[1.0, 2.0, 3.0], [1000.0, 1001.0, 1002.0]]))
encoded = one_hot(np.array([0, 2, 1]), num_classes=3)
distances = knn_distances(features[:2], features)

assert np.allclose(normalized.mean(axis=0), 0, atol=1e-10)
assert np.allclose(normalized.std(axis=0), 1, atol=1e-10)
assert similarities.shape == (10, 10)
assert np.allclose(probabilities.sum(axis=1), 1)
assert encoded.shape == (3, 3)
assert distances.shape == (2, 10)

print('normalized shape:', normalized.shape)
print('similarity diagonal:', np.diag(similarities).round(3))
print('softmax row sums:', probabilities.sum(axis=1))
print('k-NN distance matrix shape:', distances.shape)

## 2. SciPy 选学入口

原始计划把 SciPy 列为选学内容。个人 Notebook 可以在这里继续补充 `scipy.stats` 的统计检验、`scipy.spatial.distance` 的距离计算或 `scipy.signal` 的信号处理。下面只检查环境，不把 SciPy 设为本示例的必需依赖。

In [ ]:
try:
    import scipy
    from scipy import stats

    group_a = rng.normal(loc=0.5, size=30)
    group_b = rng.normal(loc=0.0, size=30)
    t_stat, p_value = stats.ttest_ind(group_a, group_b)
    print('SciPy:', scipy.__version__)
    print(f't = {t_stat:.3f}, p = {p_value:.4f}')
except ImportError:
    print('SciPy 未安装；NumPy 核心示例仍可运行。')

## 小结

向量化代码把一批样本的计算表达成数组运算，通常比逐样本循环更简洁，也更接近后续机器学习框架的写法。提交自己的 Notebook 时，应补充输入输出形状、边界情况和结果解释。